In [33]:
# RUN THIS CELL FIRST
import os
import math
from collections import OrderedDict

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import v2

import numpy as np
from numpy import allclose, isclose

from collections.abc import Callable
from sklearn.model_selection import train_test_split

ASSETS_PATH = "data/new_training"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
transformations = transforms.Compose(
    [
        v2.Resize((128,128)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.RandomHorizontalFlip(),
        v2.RandomRotation(45),
        v2.ColorJitter(brightness= 0.5, contrast = 0.5, saturation= 0.3),
        v2.RandomCrop((60,60)),
        v2.RandomInvert(p=0.3),
        v2.Normalize(mean=[0.5, 0.5, 0.5],
                                 std=[0.5, 0.5, 0.5])
    ]
)


cuda


In [34]:
dataset = datasets.ImageFolder(ASSETS_PATH, transform=transformations)
class_labels = list(dataset.class_to_idx.keys())
indices = list(range(len(dataset)))
labels = [y for _, y in dataset]
train_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=labels)

train_set = Subset(dataset, train_idx)
test_set = Subset(dataset, test_idx)

train_loader = DataLoader(train_set, batch_size=256, shuffle=True, drop_last=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=True)

print(labels)



[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [52]:
def train_model(model: nn.Module, dataloader: DataLoader, epochs: int = 20):
    """
    Trains the model for a specified number of epochs/iterations
    
    Parameters
    ---------- 
        model: A PyTorch model to be trained
        dataloader : A DataLoader object that provides batches of the training data
        epochs  : Number of epochs, default of 20
        
    Returns
    -------
        The final model and the loss curve (per epoch)
    """

    losses = []
    loss_fn = nn.CrossEntropyLoss()
    # Set model to training mode. 
    # See (https://stackoverflow.com/questions/60018578/what-does-model-eval-do-in-pytorch) if curious.
    model.train() 
    optimiser = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.01)
    scheduler = torch.optim.lr_scheduler.StepLR(optimiser, step_size=10, gamma=0.1)
    # optimiser = torch.optim.AdamW(model.parameters())
    for i in range(epochs):
        epoch_loss = 0.0
        for x_batch, y_batch in dataloader:
            optimiser.zero_grad()
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            y_output = model.predict_proba(x_batch)
            loss = loss_fn(y_output, y_batch)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()
        print(f"Epoch {i+1}/{epochs}, Loss: {epoch_loss:.4f}")
        losses.append(epoch_loss)
        scheduler.step()
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for i, data in enumerate(test_loader):
                x, y = data
                x, y = x.to(device), y.to(device)
                y_output = model.predict_proba(x)
                val_loss += loss_fn(y_output, y).item()
        print(f"Val Loss: {val_loss:.4f}")
        model.train()

    return model, losses

In [44]:
class TestCNN(nn.Module):
    def __init__(self, classes: int):
        super().__init__()
        self.conv = nn.Sequential(
                        nn.Conv2d(3, 32, (3,3), padding=1),
                        nn.MaxPool2d((2,2)),
                        nn.LeakyReLU(0.1),
                        nn.Dropout(0.5),
                        nn.Conv2d(32, 64, (3,3), padding=1),
                        nn.MaxPool2d((2,2)),
                        nn.LeakyReLU(0.1),
                        nn.Dropout(0.5),
                        nn.Conv2d(64, 128, (3,3), padding=1),
                        nn.MaxPool2d((2,2)),
                        nn.LeakyReLU(0.1),
                        nn.Dropout(0.5),
                        nn.Conv2d(128, 256, (3,3), padding=1),
                        nn.MaxPool2d((2,2)),
                        nn.LeakyReLU(0.1),
                        nn.Dropout(0.5),
                        nn.Conv2d(256, 512, (3,3), padding=1),
                        nn.LeakyReLU(0.1),
                        nn.Dropout(0.5),
                    )

        self.fc = nn.Sequential(
                        nn.Linear(512, 256),
                        nn.LeakyReLU(0.1),
                        nn.Linear(256, 128),
                        nn.LeakyReLU(0.1),
                        nn.Dropout(0.5),
                        nn.Linear(128, classes),
                    )
        self.gap = nn.AdaptiveAvgPool2d(1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """ YOUR CODE HERE """
        x = self.conv(x)
        """ YOUR CODE END HERE """
        x = self.gap(x) # GAP – do not remove this line
        """ YOUR CODE HERE """
        x = x.view(x.shape[0], -1)
        out = self.fc(x)
        """ YOUR CODE END HERE """
        return out
    
    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        out = self.forward(x)
        return torch.softmax(out, dim = 1)

In [53]:
test_model, test_losses = train_model(TestCNN(14).to(device), train_loader, epochs = 30)

Epoch 1/30, Loss: 158.3220
Val Loss: 42.2138
Epoch 2/30, Loss: 158.2698
Val Loss: 42.2013
Epoch 3/30, Loss: 158.2159
Val Loss: 42.1900
Epoch 4/30, Loss: 158.1612
Val Loss: 42.1779
Epoch 5/30, Loss: 158.1046
Val Loss: 42.1662
Epoch 6/30, Loss: 158.0422
Val Loss: 42.1537
Epoch 7/30, Loss: 157.9700
Val Loss: 42.1407
Epoch 8/30, Loss: 157.8733
Val Loss: 42.1231
Epoch 9/30, Loss: 157.7270
Val Loss: 42.1015
Epoch 10/30, Loss: 157.4317
Val Loss: 42.0499
Epoch 11/30, Loss: 157.1505
Val Loss: 42.0383
Epoch 12/30, Loss: 157.0815
Val Loss: 42.0332
Epoch 13/30, Loss: 157.0458
Val Loss: 42.0163
Epoch 14/30, Loss: 156.9508
Val Loss: 42.0115
Epoch 15/30, Loss: 156.9007
Val Loss: 41.9960
Epoch 16/30, Loss: 156.8976
Val Loss: 41.9840
Epoch 17/30, Loss: 156.8611
Val Loss: 41.9803
Epoch 18/30, Loss: 156.8730
Val Loss: 41.9815
Epoch 19/30, Loss: 156.8829
Val Loss: 41.9713
Epoch 20/30, Loss: 156.8053
Val Loss: 41.9593
Epoch 21/30, Loss: 156.8225
Val Loss: 41.9521
Epoch 22/30, Loss: 156.8057
Val Loss: 41.96

In [ ]:
test_model, test_losses2 = train_model(test_model, train_loader, epochs = 10)
test_losses += test_losses2

In [56]:
def get_accuracy(scores: torch.Tensor, labels: torch.Tensor) -> int | float:
    _, predictions = torch.max(scores, 1)
    print(predictions)
    # Per-class accuracy breakdown
    classes = torch.unique(labels)
    for cls in classes:
        mask = labels == cls
        cls_acc = (predictions[mask] == labels[mask]).float().mean().item()
        print(f"Class {class_labels[cls.item()]}: {cls_acc:.2%}")
    
    overall = (predictions == labels).float().mean().item()
    print(f"Overall: {overall:.2%}")
    return overall

In [114]:
print(classes)

['boots', 'box', 'coin', 'exit', 'floor', 'gem', 'ghost', 'human', 'key', 'lava', 'locked', 'opened', 'shield', 'wall']


In [57]:
with torch.no_grad():
    test_model.eval()
    for i, data in enumerate(test_loader):
        x, y = data
        x, y = x.to(device), y.to(device)
        pred = test_model.predict_proba(x)
        acc = get_accuracy(pred, y)
        print(f"test accuracy: {acc}")

tensor([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], device='cuda:0')
Class boots: 0.00%
Class box: 0.00%
Class coin: 0.00%
Class exit: 0.00%
Class floor: 0.00%
Class gem: 0.00%
Class ghost: 0.00%

In [59]:
from collections import Counter
labels = [label for _, label in train_set]
print(Counter(labels))


Counter({7: 1920, 6: 1500, 2: 1500, 11: 1500, 5: 1500, 8: 1320, 0: 1320, 10: 1320, 3: 1320, 12: 1140, 1: 1140, 4: 20, 13: 20, 9: 20})
